# 02 - Build fine-tuning data, arm A (tourism-filtered ILCI + Samanantar)
No GPU needed. Downloads ILCI (62 MB) and Samanantar Hindi (1.9 GB), filters tourism-like pairs,
removes everything that overlaps the test sets, and writes train/dev files + a report.

**Before running:** accept the BPCC terms while logged in to Hugging Face:
https://huggingface.co/datasets/ai4bharat/BPCC

In [ ]:
# Cell 1: Hugging Face login (BPCC and FLORES+ are gated)
from huggingface_hub import login, whoami
try:
    print("Logged in as", whoami()["name"])
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

In [ ]:
# Cell 2: get (or update) the code from GitHub
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"
if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed" || true
!git log -1 --oneline

In [ ]:
# Cell 3: build arm A (about 10-20 minutes: mostly downloading and reading 4.4M Samanantar pairs)
!python src/build_finetune_data.py --config configs/finetune_arm_a.yaml

In [ ]:
# Cell 4: show 30 random training pairs so we can judge the quality by eye
import csv, random
rows = list(csv.DictReader(open("data/finetune/arm_a/train.tsv", encoding="utf-8"), delimiter="\t"))
for r in random.Random(0).sample(rows, min(30, len(rows))):
    print(f"[{r['source']}] [{r['keywords']}]\n  EN: {r['en']}\n  HI: {r['hi']}")

In [ ]:
# Cell 5: put the report into this notebook's output so it can be saved on the laptop (then Ctrl+S)
print("===REPORT-START===")
print(open("results/finetune_data_arm_a.json", encoding="utf-8").read())
print("===REPORT-END===")